In [1]:
print("hello world")

hello world


In [6]:
import re
import os
from dotenv import load_dotenv

from atlassian import Confluence
from markdownify import markdownify as md
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from typing import List, Dict

load_dotenv()

# ========================== CONFIG ==========================
CONFLUENCE_PAGE_ID = 1671169
COLLECTION_NAME = "python_coding_standards"
DB_PATH = "./chroma_python_standards_db"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # Small, fast, excellent for technical docs

In [7]:
# ========================== FETCH FROM CONFLUENCE ==========================
def fetch_confluence_page(page_id: int) -> str:
    confluence = Confluence(
        url=os.getenv("CF_URL"),
        username=os.getenv("ACCOUNT"),
        password=os.getenv("CF_TOKEN")
    )

    page = confluence.get_page_by_id(
        page_id=page_id,
        expand='body.storage'
    )

    html_content = page['body']['storage']['value']
    title = page.get('title', 'Untitled')

    print(f"✅ Fetched page: '{title}' (ID: {page_id})")
    return html_content, title

In [8]:
# Step 1: Fetch from Confluence
html_content, page_title = fetch_confluence_page(CONFLUENCE_PAGE_ID)

✅ Fetched page: 'Test age 2' (ID: 1671169)


In [9]:
print(f"Page Title: {page_title}")
print(f"HTML Content Length: {len(html_content)} characters")

Page Title: Test age 2
HTML Content Length: 20343 characters


In [10]:
# ========================== CLEAN HTML → MARKDOWN ==========================
def html_to_clean_markdown(html_content: str, page_title: str) -> str:
    # Convert Confluence HTML to Markdown
    markdown = md(
        html_content,
        heading_style="ATX",           # Use # ## ### style
        bullets="*",                   # Consistent bullets
        code_language_callback=lambda _: "python",  # Default code blocks to python
        strip=['img', 'script', 'style'],  # Remove unnecessary tags
        convert_links=True
    )

    # Optional post-processing for better RAG quality
    markdown = re.sub(r'\n{3,}', '\n\n', markdown)   # Reduce excessive blank lines
    markdown = re.sub(r'^\s*[-*+]\s+', '* ', markdown, flags=re.MULTILINE)  # Normalize lists

    # Add front-matter like title
    final_md = f"# {page_title}\n\n{markdown.strip()}"
    
    print("✅ Converted Confluence HTML to clean Markdown")
    return final_md

In [11]:
# Step 2: Convert to clean Markdown
markdown_content = html_to_clean_markdown(html_content, page_title)

✅ Converted Confluence HTML to clean Markdown


In [12]:
print(f"Type: {type(markdown_content)}")
print(f"Markdown content: {markdown_content}")  # Print first 500 chars for verification

Type: <class 'str'>
Markdown content: # Test age 2

Here's the **updated Python Coding Standards** page for Confluence, now using **plain Markdown ASCII flow diagrams** (text-based flowcharts) instead of Mermaid or images.

These ASCII diagrams render cleanly in Confluence when pasted as code blocks or plain text.

---

# Python Coding Standards

**Document Owner:** Engineering Team   
**Version:** 1.3   
**Last Updated:** March 2026

---

## Table of Contents
* 1. Introduction
* 2. General Principles
* 3. Code Layout & Formatting
* 4. Naming Conventions
* 5. Documentation & Comments
* 6. Imports
* 7. Best Practices
* 8. Error Handling
* 9. Testing Standards
* 10. Security Considerations
* 11. Tools & Enforcement
* 12. Python Coding Workflow

---

## Introduction

This document defines the Python coding standards to be followed across all Python projects. Consistent code style improves readability, maintainability, and reduces bugs.

We follow **PEP 8** as the base standard, with addit

In [14]:
with open("python_coding_standards.md", "w", encoding="utf-8") as f:
    f.write(markdown_content)
print("✅ Saved clean Markdown to python_coding_standards.md")

✅ Saved clean Markdown to python_coding_standards.md


In [13]:
# ========================== HEADER-BASED CHUNKING ==========================
def chunk_by_headers(markdown_text: str) -> List[Dict]:
    """Best strategy: Split on headers (H1-H3) while preserving tables, code, ASCII diagrams."""
    header_pattern = re.compile(r'^(#{1,4})\s+(.*)', re.MULTILINE)
    
    chunks = []
    current_title = "Document Root"
    current_level = 0
    current_content = []
    last_pos = 0

    for match in header_pattern.finditer(markdown_text):
        start = match.start()
        level = len(match.group(1))
        title = match.group(2).strip()

        # Save previous chunk
        if current_content:
            content = "\n".join(current_content).strip()
            if content:
                chunks.append({
                    "chunk_id": f"chunk_{len(chunks)}",
                    "section_title": current_title,
                    "header_level": current_level,
                    "content": content,
                    "parent_section": chunks[-1]["section_title"] if chunks else "None",
                    "source": "Confluence_Python_Coding_Standards"
                })

        # Start new chunk
        current_title = title
        current_level = level
        current_content = [match.group(0)]
        last_pos = match.end()

    # Add final chunk
    current_content.append(markdown_text[last_pos:])
    content = "\n".join(current_content).strip()
    if content:
        chunks.append({
            "chunk_id": f"chunk_{len(chunks)}",
            "section_title": current_title,
            "header_level": current_level,
            "content": content,
            "parent_section": chunks[-1]["section_title"] if chunks else "None",
            "source": "Confluence_Python_Coding_Standards"
        })

    return chunks

In [15]:
# ========================== INGEST TO CHROMADB ==========================
def ingest_to_chroma(markdown_content: str, page_title: str):
    chunks = chunk_by_headers(markdown_content)
    print(f"✅ Created {len(chunks)} semantic chunks from the page")

    # Embedding function
    embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

    # ChromaDB client
    client = chromadb.PersistentClient(path=DB_PATH)
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function,
        metadata={"hnsw:space": "cosine"}
    )

    # Prepare for ingestion
    ids = [c["chunk_id"] for c in chunks]
    documents = [c["content"] for c in chunks]
    metadatas = [{
        "section_title": c["section_title"],
        "header_level": c["header_level"],
        "parent_section": c["parent_section"],
        "source": c["source"],
        "page_title": page_title
    } for c in chunks]

    collection.add(ids=ids, documents=documents, metadatas=metadatas)

    print("🚀 Ingestion completed successfully!")
    print(f"   Collection : {COLLECTION_NAME}")
    print(f"   Chunks     : {len(chunks)}")
    print(f"   Model      : {EMBEDDING_MODEL}")
    print(f"   DB Path    : {DB_PATH}")

In [16]:
# Step 3: Ingest into ChromaDB
ingest_to_chroma(markdown_content, page_title)

✅ Created 23 semantic chunks from the page


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\dharm\Downloads\code_reviewer\get-diff\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dharm\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

🚀 Ingestion completed successfully!
   Collection : python_coding_standards
   Chunks     : 23
   Model      : all-MiniLM-L6-v2
   DB Path    : ./chroma_python_standards_db
